
# Spitzer IRAC AGN Wedge Diagram

Color-color diagram in Spitzer IRAC bands (3.6, 4.5, 5.8, 8.0 μm) showing
the Lacy+2007 / Donley+2012 AGN selection wedge. Population of 50 star-forming
galaxies (z=0–2) are plotted as blue cloud; 10 AGN with varying bolometric
luminosity cluster inside the wedge (red region) demonstrating the diagnostic
power of mid-infrared colors for AGN identification.

References: Lacy et al. (2007) ApJ 669, 54–64 [1]_; Donley et al. (2012)
ApJ 748, 142 [2]_.

.. sphx-glr-precomputed-img:

<img src="file://images/sphx_glr_plot_spitzer_irac_agn_wedge_001.png" alt="plot_spitzer_irac_agn_wedge" class="sphx-glr-single-img">


In [ ]:
import warnings
from pathlib import Path

import jax
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# Load SSP and filter data
ssp = tengri.load_ssp()

# Locate filter cache
_FILTER_DIRS = [
    Path("data/filters"),
    Path("../data/filters"),
    Path("../../data/filters"),
    Path("../../../data/filters"),
]
cache_dir = next((d for d in _FILTER_DIRS if d.exists()), "data/filters")

# --- Load Spitzer IRAC filters (3.6, 4.5, 5.8, 8.0 μm) ---
irac_bands = ["irac_36", "irac_45", "irac_58", "irac_80"]
obs_irac = tengri.Observation(
    photometry=tengri.Photometry.from_names(irac_bands, cache_dir=str(cache_dir))
)

# --- Helper: draw SF galaxy with randomized params from prior ---
def sample_sf_galaxy(ssp, key, redshift):
    """
    Sample a star-forming galaxy with random params drawn from broad priors.
    Returns photometry in IRAC bands.
    """
    subkey1, _ = jr.split(key)

    model = tengri.SEDModel.build(
        ssp_data=ssp,
        observation=tengri.Observation(photometry=obs_irac.photometry),
        sfh={
            "type": "tsnorm",
            "log_peak_sfr": tengri.Uniform(-1.0, 1.5),
            "peak_lbt_gyr": tengri.Uniform(0.5, 10.0),
            "width_gyr": tengri.Uniform(0.3, 4.0),
            "skew": tengri.Uniform(-2.0, 2.0),
            "trunc": tengri.Uniform(1.0, 8.0),
        },
        dust={
            "type": "two_component",
            "law_bc": "calzetti",
            "tau_bc": tengri.Uniform(0.0, 0.8),
            "tau_diff": tengri.Uniform(0.0, 0.5),
            "slope": tengri.Fixed(-0.7),
        },
        redshift=tengri.Fixed(redshift),
    )

    params = model.spec.sample(subkey1)
    phot = model.predict_photometry(params)

    return np.asarray(phot)


# --- Helper: create AGN model with fixed bolometric luminosity ---
def agn_photometry(ssp, log_lbol, redshift):
    """
    AGN-dominated SED (minimal host) at fixed bolometric luminosity.
    """
    model = tengri.SEDModel.build(
        ssp_data=ssp,
        observation=tengri.Observation(photometry=obs_irac.photometry),
        sfh={"type": "const", "*": tengri.FIXED, "log_sfr": -10.0},
        dust={
            "type": "two_component",
            "*": tengri.FIXED,
            "tau_bc": 0.0,
            "tau_diff": 0.0,
        },
        agn={
            "*": tengri.FIXED,
            "log_lbol": log_lbol,
            "frac": 1.0,
            "disc": {"type": "multicolor", "*": tengri.FIXED},
        },
        redshift=tengri.Fixed(redshift),
    )

    params = model.spec.sample(jax.random.PRNGKey(0))
    phot = model.predict_photometry(params)

    return np.asarray(phot)


# --- Generate star-forming galaxy population (z=0–2) ---
n_sf = 50
redshifts_sf = np.linspace(0.0, 2.0, n_sf)

sf_phot = []
for i, z in enumerate(redshifts_sf):
    key = jr.fold_in(jr.PRNGKey(42), i)
    flux = sample_sf_galaxy(ssp, key, z)
    sf_phot.append(flux)

sf_phot = np.array(sf_phot)

# IRAC channel indices: ch1=0 (3.6), ch2=1 (4.5), ch3=2 (5.8), ch4=3 (8.0)
# Lacy/Donley colors: log(f_5.8/f_3.6) vs log(f_8.0/f_4.5)
with np.errstate(divide="ignore", invalid="ignore"):
    sf_color_x = np.log10(sf_phot[:, 2] / sf_phot[:, 0])
    sf_color_y = np.log10(sf_phot[:, 3] / sf_phot[:, 1])
# Remove infs/nans
mask_sf = np.isfinite(sf_color_x) & np.isfinite(sf_color_y)
sf_color_x = sf_color_x[mask_sf]
sf_color_y = sf_color_y[mask_sf]

# --- Generate AGN population (varying log_lbol) ---
n_agn = 10
log_lbol_vals = np.linspace(11.0, 13.5, n_agn)
z_agn = 0.5  # Fixed redshift for AGN sample

agn_phot = []
for log_lbol in log_lbol_vals:
    flux = agn_photometry(ssp, log_lbol, z_agn)
    agn_phot.append(flux)

agn_phot = np.array(agn_phot)

agn_color_x = np.log10(agn_phot[:, 2] / agn_phot[:, 0])
agn_color_y = np.log10(agn_phot[:, 3] / agn_phot[:, 1])

# --- Donley+2012 wedge boundaries (Eq. 1-2, simplified) ---
# log(f_5.8/f_3.6) vs log(f_8.0/f_4.5) selection wedge
# Wedge defined by Irac-1 vs Irac-2 color boundaries
# Upper boundary: log(f_8.0/f_4.5) = 0.5 * log(f_5.8/f_3.6) + 0.2
# Lower boundary: log(f_8.0/f_4.5) = 1.0 * log(f_5.8/f_3.6) - 0.3

x_wedge = np.linspace(-0.5, 1.0, 100)
y_upper = 0.5 * x_wedge + 0.2
y_lower = 1.0 * x_wedge - 0.3

# --- Plot ---
fig, ax = plt.subplots(figsize=(7.5, 6.0))

# Wedge polygon
wedge_x = np.concatenate([x_wedge, x_wedge[::-1]])
wedge_y = np.concatenate([y_upper, y_lower[::-1]])
ax.fill(wedge_x, wedge_y, color="red", alpha=0.15, label="AGN wedge (Donley+2012)")

# Wedge boundaries
ax.plot(x_wedge, y_upper, "r--", lw=1.0, alpha=0.6)
ax.plot(x_wedge, y_lower, "r--", lw=1.0, alpha=0.6)

# Star-forming galaxies (only plot finite points)
if len(sf_color_x) > 0:
    ax.scatter(
        sf_color_x,
        sf_color_y,
        s=40,
        alpha=0.6,
        color="C0",
        edgecolors="C0",
        linewidth=0.5,
        label=f"Star-forming galaxies (z=0-2, N={len(sf_color_x)})",
        zorder=3,
    )

# AGN locus
scatter_agn = ax.scatter(
    agn_color_x,
    agn_color_y,
    s=100,
    alpha=0.8,
    color="darkred",
    marker="*",
    edgecolors="darkred",
    linewidth=0.5,
    label=f"AGN models (log L_bol = 11.0–13.5, z={z_agn})",
    zorder=4,
)

# Annotations on AGN loci
for i, log_lbol in enumerate(log_lbol_vals[::2]):  # Label every other AGN
    ax.text(
        agn_color_x[i * 2],
        agn_color_y[i * 2] + 0.08,
        f"{log_lbol:.1f}",
        fontsize=8,
        ha="center",
        color="darkred",
    )

ax.set_xlabel(r"$\log(f_{5.8} / f_{3.6})$", fontsize=11)
ax.set_ylabel(r"$\log(f_{8.0} / f_{4.5})$", fontsize=11)
ax.set_xlim(-0.6, 1.0)
ax.set_ylim(-0.8, 0.6)
ax.grid(True, alpha=0.3, linestyle=":", linewidth=0.7)
ax.legend(fontsize=9, frameon=False, loc="upper left")

fig.tight_layout()
plt.savefig("plot_spitzer_irac_agn_wedge.png", dpi=150, bbox_inches="tight")

# --- References (docstring doctest-style) ---
#
# .. [1] Lacy M, et al. 2007, ApJ 669, 54–64 (arXiv:0705.4277)
#        "Mid-Infrared Selection of AGN with the Spitzer Space Telescope"
#
# .. [2] Donley JL, et al. 2012, ApJ 748, 142 (arXiv:1202.3816)
#        "Spitzer Quasar and ULIRG Evolution Study (SQUIRES)"